# Aula 04 — Laboratório: Métricas de regressão

Cada tópico traz um **Exemplo** já resolvido e, em seguida, um **Exercício de aplicação** para você fazer. Vários exercícios **relembram** conceitos das aulas anteriores (limpeza, EDA, treino/teste, escalonamento, correlação, classificação × regressão).

**Base:** `3_jogadores.csv` — prever o **valor de mercado** (`valor_mercado_milhoes`) de jogadores.

## Parte 0 — Ambiente, dados e um modelo

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.dummy import DummyRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv('3_jogadores.csv')
NUM = ['idade','altura','jogos','minutos','gols','assistencias','finalizacoes',
       'passes_certos_pct','dribles_certos','desarmes','interceptacoes',
       'duelos_ganhos_pct','velocidade','indice_popularidade']
ALVO = 'valor_mercado_milhoes'
X, y = df[NUM], df[ALVO]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
modelo = LinearRegression().fit(X_train, y_train)
pred = modelo.predict(X_test)      # previsões no CONJUNTO DE TESTE
print('Modelo treinado. R² no teste:', round(r2_score(y_test, pred), 3))

## 1. Revisão: qualidade dos dados (limpeza e EDA)

**Exemplo 1.** antes de modelar, sempre checamos a base: faltantes, duplicatas e um resumo do alvo.

In [ ]:
print('faltantes:', int(df.isna().sum().sum()), '| duplicatas:', int(df.duplicated().sum()))
print(df[ALVO].describe().round(2))

**Exercício 1 (aplicação).** Conte quantos jogadores há por `posicao` e mostre o **valor de mercado médio** por posição (dica: `groupby`).

In [ ]:
# TODO — resolva aqui


## 2. Revisão: treino e teste (avaliar onde o modelo não treinou)

**Exemplo 2.** separamos os dados: o modelo aprende no treino e é avaliado no teste (que ele nunca viu).

In [ ]:
print('treino:', X_train.shape, '| teste:', X_test.shape)

**Exercício 2 (aplicação).** Refaça a divisão com `test_size=0.2` e `random_state=1` e informe os tamanhos de treino e teste.

In [ ]:
# TODO — resolva aqui


## 3. Treinar um modelo de regressão e prever

**Exemplo 3.** um regressor recebe as features e devolve um **número** (não uma classe).

In [ ]:
reg = LinearRegression().fit(X_train, y_train)
print('5 primeiras previsões no teste:', reg.predict(X_test)[:5].round(2))

**Exercício 3 (aplicação).** Mostre as 5 primeiras previsões **ao lado** do valor real (`y_test`).

In [ ]:
# TODO — resolva aqui


## 4. Erro (resíduo) = valor real − valor previsto

**Exemplo 4.** o erro de cada previsão é o real menos o previsto (positivo = subestimou).

In [ ]:
reais = np.array([30, 25, 40.]); prev = np.array([28, 27, 35.])
print('erros:', reais - prev)

**Exercício 4 (aplicação).** Calcule os resíduos (`y_test − pred`) das **5 primeiras** previsões do teste.

In [ ]:
# TODO — resolva aqui


## 5. MAE — erro absoluto médio

**Exemplo 5.** MAE = média dos erros sem sinal (em módulo). Lê-se na unidade do alvo.

In [ ]:
erros = np.array([5, 3, 2, 6, 44.])
print('MAE =', np.abs(erros).mean())

**Exercício 5 (aplicação).** Calcule o **MAE** do modelo no teste com `mean_absolute_error`.

In [ ]:
# TODO — resolva aqui


## 6. MSE — erro quadrático médio

**Exemplo 6.** MSE = média dos erros AO QUADRADO (fica na unidade² do alvo).

In [ ]:
print('MSE =', (erros**2).mean())

**Exercício 6 (aplicação).** Calcule o **MSE** do modelo no teste com `mean_squared_error`.

In [ ]:
# TODO — resolva aqui


## 7. RMSE — pune com força os erros grandes

**Exemplo 7.** RMSE = raiz do MSE (volta à unidade do alvo). O erro de 44 domina.

In [ ]:
print('MAE =', np.abs(erros).mean(), '| RMSE =', round(np.sqrt((erros**2).mean()),2))

**Exercício 7 (aplicação).** Calcule o **RMSE** no teste e compare com o MAE. Qual é maior e por quê?

In [ ]:
# TODO — resolva aqui


## 8. R² — coeficiente de determinação (na mão e pronto)

**Exemplo 8.** R² = 1 − SSres/SStot: SSres = soma dos erros² do modelo; SStot = soma dos desvios² da média.

In [ ]:
yv = np.array([2,5,6,8,9.]); yh = np.array([2.6,4.3,6,7.7,9.4])
SSres = ((yv-yh)**2).sum(); SStot = ((yv-yv.mean())**2).sum()
print('R² na mão =', round(1-SSres/SStot,4), '| r2_score =', round(r2_score(yv,yh),4))

**Exercício 8 (aplicação).** Calcule o **R²** do modelo no teste com `r2_score` e confirme na mão (1 − SSres/SStot).

In [ ]:
# TODO — resolva aqui


## 9. R² = r² do Pearson (a dúvida da aula)

**Exemplo 9.** em regressão linear SIMPLES (1 preditor), o R² do modelo é EXATAMENTE o quadrado do Pearson. E um modelo ruim dá R² negativo, enquanto o Pearson entre as variáveis não muda.

In [ ]:
Xe = np.array([1,2,3,4,5.]); Ye = np.array([2,5,6,8,9.])
r = np.corrcoef(Xe, Ye)[0,1]
reg = LinearRegression().fit(Xe.reshape(-1,1), Ye)
print('Pearson r =', round(r,4), '| r² =', round(r**2,4),
      '| R² da regressão =', round(r2_score(Ye, reg.predict(Xe.reshape(-1,1))),4))
print('Modelo ruim (prevê sempre 3): R² =', round(r2_score(Ye, np.full(5,3.0)),3),
      '| Pearson continua', round(r,3))

**Exercício 9 (aplicação).** No dataset, use só a feature `indice_popularidade` para prever o alvo: calcule o **Pearson r** entre elas, o **r²**, e o **R²** de uma regressão linear simples com essa feature. Eles batem?

In [ ]:
# TODO — resolva aqui


## 10. Baseline: prever sempre a média (R² = 0)

**Exemplo 10.** o R² compara o modelo com o 'chute da média'. Prever sempre a média dá R² = 0.

In [ ]:
dummy = DummyRegressor(strategy='mean').fit(X_train, y_train)
print('R² do baseline (média):', round(r2_score(y_test, dummy.predict(X_test)),3))

**Exercício 10 (aplicação).** Compare o **R²** do seu modelo com o do **DummyRegressor**. O modelo é melhor que chutar a média?

In [ ]:
# TODO — resolva aqui


## 11. MAPE — erro em porcentagem

**Exemplo 11.** MAPE = média de (|erro| ÷ valor real) × 100%. Não depende do tamanho do alvo.

In [ ]:
real = np.array([30, 40.]); prv = np.array([33, 38.])
print('MAPE =', round((np.abs(real-prv)/real).mean()*100,1), '%')

**Exercício 11 (aplicação).** Calcule o **MAPE** do modelo no teste (em %).

In [ ]:
# TODO — resolva aqui


## 12. A armadilha do MAPE (valores perto de zero)

**Exemplo 12.** quando o valor real é pequeno, o percentual explode: errar 5 sobre 5 = 100%.

In [ ]:
print('erro de 5 sobre real 5 ->', 5/5*100, '%  | erro de 5 sobre real 200 ->', round(5/200*100,1), '%')

**Exercício 12 (aplicação).** Quantos jogadores têm valor de mercado **abaixo de 1 milhão**? Explique por que isso torna o MAPE instável nesta base.

In [ ]:
# TODO — resolva aqui


## 13. Métricas com unidade × sem unidade

**Exemplo 13.** MAE/RMSE vêm na unidade do alvo (não comparam alvos diferentes); R² e MAPE são adimensionais.

In [ ]:
print('MAE (milhões):', round(mean_absolute_error(y_test,pred),2),
      '| R² (fração):', round(r2_score(y_test,pred),3))

**Exercício 13 (aplicação).** Explique (com um `print`) por que **não** dá para comparar o MAE deste modelo (em milhões de €) com o MAE de um modelo que prevê idade (em anos), e qual métrica usar para comparar.

In [ ]:
# TODO — resolva aqui


## 14. Gráfico de resíduos (onde o modelo erra)

**Exemplo 14.** plotar o resíduo contra o valor previsto ajuda a ver padrões (o ideal é uma nuvem sem forma).

In [ ]:
res_ex = y_test.values - pred
plt.figure(figsize=(6,3)); plt.axhline(0, color='gray')
plt.scatter(pred, res_ex, s=8, alpha=.5); plt.xlabel('previsto'); plt.ylabel('resíduo')
plt.title('Resíduos × previsto'); plt.tight_layout(); plt.show()

**Exercício 14 (aplicação).** Faça o mesmo gráfico e diga: os resíduos crescem para valores previstos altos? (isso indicaria que o modelo erra mais nos jogadores mais caros).

In [ ]:
# TODO — resolva aqui


## 15. Overfitting: erro no treino × no teste

**Exemplo 15.** se o modelo vai muito melhor no treino do que no teste, ele decorou (overfitting).

In [ ]:
print('R² treino:', round(r2_score(y_train, modelo.predict(X_train)),3),
      '| R² teste:', round(r2_score(y_test, pred),3))

**Exercício 15 (aplicação).** Treine uma **árvore de decisão sem limite** (`DecisionTreeRegressor()`), compare o R² de treino e de teste e diga se houve overfitting.

In [ ]:
# TODO — resolva aqui


## 16. Comparar dois modelos pelas métricas

**Exemplo 16.** para escolher, comparamos os modelos com as MESMAS métricas no MESMO teste.

In [ ]:
for nome, mdl in [('Linear', LinearRegression()), ('Árvore', DecisionTreeRegressor(max_depth=4, random_state=0))]:
    mdl.fit(X_train, y_train); pv = mdl.predict(X_test)
    print(f'{nome:8s} MAE={mean_absolute_error(y_test,pv):.2f}  R²={r2_score(y_test,pv):.3f}')

**Exercício 16 (aplicação).** Adicione o `DummyRegressor` (baseline) à comparação e diga qual modelo você escolheria e por quê.

In [ ]:
# TODO — resolva aqui


## 17. Revisão: covariância e correlação de Pearson

**Exemplo 17.** a correlação mede a relação linear entre duas variáveis (−1 a 1). Útil para ver o que 'puxa' o alvo.

In [ ]:
print('cov(gols, alvo)=', round(df['gols'].cov(df[ALVO]),2),
      '| r(gols, alvo)=', round(df['gols'].corr(df[ALVO]),3))

**Exercício 17 (aplicação).** Liste as **3 features mais correlacionadas** (em módulo) com o alvo.

In [ ]:
# TODO — resolva aqui


## 18. Revisão: redundância entre features (correlação alta)

**Exemplo 18.** duas features muito correlacionadas carregam a mesma informação (redundância).

In [ ]:
c = df[NUM].corr().abs()
import numpy as np; np.fill_diagonal(c.values, 0)
print('par mais correlacionado:', c.stack().idxmax(), '| |r| =', round(c.stack().max(),2))

**Exercício 18 (aplicação).** Encontre o par de features com **maior |r|** e diga o que você faria com ele (dica: manter uma).

In [ ]:
# TODO — resolva aqui


## 19. Revisão: escalonar sem vazamento (Pipeline)

**Exemplo 19.** o scaler deve aprender SÓ no treino; o Pipeline garante isso automaticamente na validação.

In [ ]:
pipe = make_pipeline(StandardScaler(), LinearRegression()).fit(X_train, y_train)
print('R² (pipeline, teste):', round(r2_score(y_test, pipe.predict(X_test)),3))

**Exercício 19 (aplicação).** Monte um `Pipeline` com `StandardScaler` + `LinearRegression`, treine no treino e avalie no teste.

In [ ]:
# TODO — resolva aqui


## 20. Classificação × regressão: a métrica certa

**Exemplo 20.** prever um NÚMERO (valor) é regressão → use MAE/RMSE/R². Prever uma CLASSE (caro/barato) é classificação → use acurácia/precisão/recall (Aulas 2 e 3).

In [ ]:
print('Prever valor_mercado (número)  -> regressão -> MAE, RMSE, R²')
print('Prever se o valor > 5 milhões (sim/não) -> classificação -> acurácia, precisão, recall')

**Exercício 20 (aplicação).** Crie o alvo de CLASSIFICAÇÃO `caro = (valor_mercado_milhoes > 5)` e diga qual métrica você usaria para avaliá-lo — e por que a acurácia NÃO serve para o alvo original (o número).

In [ ]:
# TODO — resolva aqui


## 21. Onde o modelo erra mais (análise por faixa)

**Exemplo 21.** comparar o erro em faixas do alvo mostra se o modelo é pior em certos valores.

In [ ]:
faixa = pd.cut(y_test, bins=[0,2,5,20], labels=['barato','médio','caro'])
print(pd.Series(np.abs(y_test.values-pred), index=y_test.index).groupby(faixa, observed=True).mean().round(2))

**Exercício 21 (aplicação).** Calcule o **MAE por faixa** (barato/médio/caro) e diga em qual faixa o modelo erra mais.

In [ ]:
# TODO — resolva aqui


## 22. Síntese: qual métrica reportar

**Exemplo 22.** reporte quase sempre um PAR: um erro na unidade (MAE ou RMSE) + o R². O MAPE só se não houver valores perto de zero.

In [ ]:
print(f'MAE={mean_absolute_error(y_test,pred):.2f} milhões | R²={r2_score(y_test,pred):.3f}')

**Exercício 22 (aplicação).** Escreva **três conclusões** deste laboratório: (a) o que o R² do modelo tem a ver com o r² do Pearson; (b) por que o MAPE é arriscado nesta base; (c) qual par de métricas você reportaria e por quê.

In [ ]:
# TODO — resolva aqui
